# Ensemble/Voting Classification in Python with Scikit-Learn
ref：https://www.kaggle.com/c/titanic/submit

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, log_loss
from sklearn.model_selection import train_test_split, KFold, cross_val_score

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier, ExtraTreesClassifier

In [2]:
training_data = pd.read_csv('data/train.csv')
testing_data = pd.read_csv('data/test.csv')

def get_nulls(training, testing):
    print("Training Data:")
    print(pd.isnull(training).sum())
    print("Testing Data:")
    print(pd.isnull(testing).sum())

get_nulls(training_data, testing_data)

Training Data:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
Testing Data:
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [3]:
# Drop columns that are too sparse or too high-cardinality for this exercise.
columns_to_drop = ['Cabin', 'Ticket', 'Name']
training_data = training_data.drop(columns=columns_to_drop)
testing_data = testing_data.drop(columns=columns_to_drop)

# Use median/mode imputations to avoid skew-sensitive mean filling.
training_data['Age'] = training_data['Age'].fillna(training_data['Age'].median())
testing_data['Age'] = testing_data['Age'].fillna(testing_data['Age'].median())
testing_data['Fare'] = testing_data['Fare'].fillna(testing_data['Fare'].median())
training_data['Embarked'] = training_data['Embarked'].fillna(training_data['Embarked'].mode()[0])
testing_data['Embarked'] = testing_data['Embarked'].fillna(testing_data['Embarked'].mode()[0])

get_nulls(training_data, testing_data)

Training Data:
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64
Testing Data:
PassengerId    0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64


In [4]:
# Encode Sex.
encoder_1 = LabelEncoder()
encoder_1.fit(training_data["Sex"])
training_data["Sex"] = encoder_1.transform(training_data["Sex"])
testing_data["Sex"] = encoder_1.transform(testing_data["Sex"])

# Encode Embarked.
encoder_2 = LabelEncoder()
encoder_2.fit(pd.concat([training_data["Embarked"], testing_data["Embarked"]], ignore_index=True))
training_data["Embarked"] = encoder_2.transform(training_data["Embarked"])
testing_data["Embarked"] = encoder_2.transform(testing_data["Embarked"])

# Scale Age and Fare.
ages_train = np.array(training_data["Age"]).reshape(-1, 1)
ages_test = np.array(testing_data["Age"]).reshape(-1, 1)
fares_train = np.array(training_data["Fare"]).reshape(-1, 1)
fares_test = np.array(testing_data["Fare"]).reshape(-1, 1)

age_scaler = StandardScaler()
fare_scaler = StandardScaler()

training_data["Age"] = age_scaler.fit_transform(ages_train)
testing_data["Age"] = age_scaler.transform(ages_test)
training_data["Fare"] = fare_scaler.fit_transform(fares_train)
testing_data["Fare"] = fare_scaler.transform(fares_test)

training_data.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,1,-0.565736,1,0,-0.502445,2
1,2,1,1,0,0.663861,1,0,0.786845,0
2,3,1,3,0,-0.258337,0,0,-0.488854,2
3,4,1,1,0,0.433312,1,0,0.420730,2
4,5,0,3,1,0.433312,0,0,-0.486337,2


In [5]:
# Now to select our training/testing data
X_features = training_data.drop(labels=['PassengerId', 'Survived'], axis=1)
y_labels = training_data['Survived']

print(X_features.head(5))
print(y_labels.head(5))

# Make the train/test data from validation

X_train, X_val, y_train, y_val = train_test_split(X_features, y_labels, test_size=0.1,random_state=12)

   Pclass  Sex       Age  SibSp  Parch      Fare  Embarked
0       3    1 -0.565736      1      0 -0.502445         2
1       1    0  0.663861      1      0  0.786845         0
2       3    0 -0.258337      0      0 -0.488854         2
3       1    0  0.433312      1      0  0.420730         2
4       3    1  0.433312      0      0 -0.486337         2
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64


## Simple Averaging Approach

In [6]:
LogReg_clf = LogisticRegression(max_iter=1000, random_state=12)
DTree_clf = DecisionTreeClassifier(random_state=12)
SVC_clf = SVC()

LogReg_clf.fit(X_train, y_train)
DTree_clf.fit(X_train, y_train)
SVC_clf.fit(X_train, y_train)

LogReg_pred = LogReg_clf.predict(X_val)
DTree_pred = DTree_clf.predict(X_val)
SVC_pred = SVC_clf.predict(X_val)

averaged_preds = ((LogReg_pred + DTree_pred + SVC_pred) / 3 >= 0.5).astype(int)
acc = accuracy_score(y_val, averaged_preds)
print(acc)

0.8333333333333334


## Bagging Classification Example

In [7]:
def bagging_ensemble(model):
    k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
    results = cross_val_score(model, X_train, y_train, cv=k_folds)
    print(results.mean())

bagging_ensemble(BaggingClassifier(estimator=LogisticRegression(max_iter=1000), n_estimators=20, random_state=12))
bagging_ensemble(BaggingClassifier(estimator=DecisionTreeClassifier(random_state=12), n_estimators=20, random_state=12))
bagging_ensemble(BaggingClassifier(estimator=SVC(), n_estimators=20, random_state=12))

0.7964634146341464


0.8138719512195122


0.8264329268292683


## Boosting Classification Example

In [8]:
k_folds = KFold(n_splits=20, random_state=12,shuffle=True)
num_estimators = [20, 40, 60, 80, 100]

for i in num_estimators:
    ada_boost = AdaBoostClassifier(n_estimators=i, random_state=12)
    results = cross_val_score(ada_boost, X_train, y_train, cv=k_folds)
    print("Results for {} estimators:".format(i))
    print(results.mean())

Results for 20 estimators:
0.8064634146341463


Results for 40 estimators:
0.8089024390243903


Results for 60 estimators:
0.8051829268292684


Results for 80 estimators:
0.8039329268292683


Results for 100 estimators:
0.8051829268292684


## voting\Stacking Classification Example

In [9]:
voting_clf = VotingClassifier(estimators=[('SVC', SVC_clf), ('DTree', DTree_clf), ('LogReg', LogReg_clf)], voting='hard')
voting_clf.fit(X_train, y_train)
preds = voting_clf.predict(X_val)
acc = accuracy_score(y_val, preds)
l_loss = log_loss(y_val, preds)
f1 = f1_score(y_val, preds)

print("Accuracy is: " + str(acc))
print("Log Loss is: " + str(l_loss))
print("F1 Score is: " + str(f1))

Accuracy is: 0.8333333333333334
Log Loss is: 6.0072755648528595
F1 Score is: 0.7761194029850746
